In [ ]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from ultralytics import YOLO
from transformers import AutoProcessor, AutoModel

In [ ]:
VALID_VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")

In [ ]:
yolo_model = YOLO("../models/yolov8x.pt")

siglip_model_name = "google/siglip-base-patch16-224"
siglip_model = AutoModel.from_pretrained(siglip_model_name)
siglip_processor = AutoProcessor.from_pretrained(siglip_model_name)

In [ ]:
video_path = "../data/video_samples/The CCTV People Demo 6.mp4"

In [ ]:
if not video_path.lower().endswith(VALID_VIDEO_EXTS):
    raise ValueError("Unsupported video format")

In [ ]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Cannot open video")

In [ ]:
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

In [ ]:
print(f"fps: {fps} \ntotalframes: {total_frames} \nwidth: {width} \nheight: {height}"
)

In [ ]:
if fps <= 0:
    fps = 30.0

In [ ]:
sample_interval_sec = 0.5
step = int(round(fps * sample_interval_sec))
step

In [ ]:
metadata_rows = []
crop_embeddings = []
frame_embeddings = []
seen_track_ids = set()

In [ ]:
current_frame = 0
row_id = 0
processed_frames = 0

In [ ]:
# while current_frame < total_frames:
#     cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
#     ret, frame = cap.read()
#     current_frame += step


In [ ]:
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("ERROR: OpenCV found the file, but cannot open it. Your OpenCV installation might be missing video codecs.")
else:
    current_frame = 0
    step = 30
    processed_frames = 0
    
    while current_frame < total_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
        ret, frame = cap.read()
        
        if not ret:
            break
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.imshow(frame_rgb)
        ax.axis('off')
        
        display(fig)
        plt.close(fig)
        current_frame += step
        processed_frames += 1
    cap.release()
    print(f"proccessed frames :{processed_frames}")

In [ ]:
current_frame = 0
row_id = 0
processed_frames = 0

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
frameno = 0

while current_frame < total_frames:
    print("====================================================")
    print("current_frame:", current_frame, " / total_frames:", total_frames)

    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()

    if not ret:
        print("Could not read frame. Stopping.")
        break

    frame_no = current_frame
    timestamp_sec = frame_no / fps

    hrs = int(timestamp_sec // 3600)
    mins = int((timestamp_sec % 3600) // 60)
    secs = int(timestamp_sec % 60)
    timestamp_str = f"{hrs:02d}:{mins:02d}:{secs:02d}"

    print("Timestamp:", timestamp_str)

    # Run YOLO + ByteTrack
    results = yolo_model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        verbose=False
    )

    if not results:
        print("No results returned by YOLO.")
        current_frame += step
        continue

    result = results[0]

    if result.boxes is None or len(result.boxes) == 0:
        print("No detections in this frame.")
        current_frame += step
        continue

    # Make a copy of frame for drawing boxes
    annotated_frame = frame.copy()

    print(f"Total detections in this frame: {len(result.boxes)}")

    for i, box in enumerate(result.boxes):
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        confidence = float(box.conf.item())

        # track id may be missing
        if box.id is None:
            track_id = None
        else:
            track_id = int(box.id.item())

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # print detection info
        print(f"\nDetection {i+1}")
        print("class_name :", class_name)
        print("confidence :", round(confidence, 4))
        print("track_id   :", track_id)
        print("bbox       :", (x1, y1, x2, y2))

        # draw RED bounding box
        cv2.rectangle(
            annotated_frame,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),   # red in BGR
            2
        )

        # label text
        label = f"{class_name} | conf:{confidence:.2f} | id:{track_id}"

        # put label above box
        cv2.putText(
            annotated_frame,
            label,
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),   # red text
            2
        )

    # Convert BGR -> RGB for matplotlib display
    annotated_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)

    # Display full frame with boxes
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(annotated_rgb)
    ax.set_title(f"Frame {frame_no} | {timestamp_str}")
    ax.axis("off")
    display(fig)
    plt.close(fig)

    current_frame += step
    frameno += 1

cap.release()
print("Total sampled frames processed:", frameno)

In [ ]:
current_frame = 0
row_id = 0
processed_frames = 0

In [ ]:
import cv2
import matplotlib.pyplot as plt

cap = cv2.VideoCapture(video_path)
frameno = 0

while current_frame < total_frames:
    print("====================================================")
    print("current_frame:", current_frame, " / total_frames:", total_frames)

    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()

    if not ret:
        print("Could not read frame. Stopping.")
        break

    frame_no = current_frame
    timestamp_sec = frame_no / fps

    hrs = int(timestamp_sec // 3600)
    mins = int((timestamp_sec % 3600) // 60)
    secs = int(timestamp_sec % 60)
    timestamp_str = f"{hrs:02d}:{mins:02d}:{secs:02d}"

    print("Timestamp:", timestamp_str)

    # Run YOLO + ByteTrack
    results = yolo_model(frame, verbose=False)

    if not results:
        print("No results returned by YOLO.")
        current_frame += step
        continue

    result = results[0]

    if result.boxes is None or len(result.boxes) == 0:
        print("No detections in this frame.")
        current_frame += step
        continue

    # Make a copy of frame for drawing boxes
    annotated_frame = frame.copy()

    print(f"Total detections in this frame: {len(result.boxes)}")

    for i, box in enumerate(result.boxes):
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        confidence = float(box.conf.item())

        # track id may be missing
        if box.id is None:
            track_id = None
        else:
            track_id = int(box.id.item())

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        # print detection info
        print(f"\nDetection {i+1}")
        print("class_name :", class_name)
        print("confidence :", round(confidence, 4))
        print("track_id   :", track_id)
        print("bbox       :", (x1, y1, x2, y2))

        # draw RED bounding box
        cv2.rectangle(
            annotated_frame,
            (x1, y1),
            (x2, y2),
            (0, 0, 255),   # red in BGR
            2
        )

        # label text
        label = f"{class_name} | conf:{confidence:.2f} | id:{track_id}"

        # put label above box
        cv2.putText(
            annotated_frame,
            label,
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),   # red text
            2
        )

    # Convert BGR -> RGB for matplotlib display
    annotated_rgb = cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB)

    # Display full frame with boxes
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(annotated_rgb)
    ax.set_title(f"Frame {frame_no} | {timestamp_str}")
    ax.axis("off")
    display(fig)
    plt.close(fig)

    current_frame += step
    frameno += 1

cap.release()
print("Total sampled frames processed:", frameno)

In [ ]:
current_frame = 0
row_id = 0
processed_frames = 0

In [ ]:
import cv2
import matplotlib.pyplot as plt

video_path = "../data/video_samples/The CCTV People Demo 6.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

if fps <= 0:
    fps = 30

sample_interval_sec = 0.5
step = max(1, int(round(fps * sample_interval_sec)))

current_frame = 0

# IMPORTANT: use a LOW threshold for debugging
debug_conf = 0.10

while current_frame < total_frames:
    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()
    if not ret:
        break

    timestamp_sec = current_frame / fps

    hrs = int(timestamp_sec // 3600)
    mins = int((timestamp_sec % 3600) // 60)
    secs = int(timestamp_sec % 60)
    timestamp_str = f"{hrs:02d}:{mins:02d}:{secs:02d}"

    # plain YOLO detect only, no tracking
    results = yolo_model(frame, conf=debug_conf, verbose=False)

    frame_vis = frame.copy()

    if results and len(results) > 0:
        result = results[0]

        if result.boxes is not None and len(result.boxes) > 0:
            print(f"\nFrame {current_frame} | {timestamp_str}")

            for box in result.boxes:
                class_id = int(box.cls.item())
                confidence = float(box.conf.item())
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                label = result.names[class_id]

                print(f"  {label} | conf={confidence:.3f} | box=({x1},{y1},{x2},{y2})")

                # draw red box
                cv2.rectangle(frame_vis, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(
                    frame_vis,
                    f"{label} {confidence:.2f}",
                    (x1, max(20, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 0, 0),
                    2
                )
        else:
            print(f"\nFrame {current_frame} | {timestamp_str} -> No detections")
    else:
        print(f"\nFrame {current_frame} | {timestamp_str} -> No detections")

    # show image
    frame_vis_rgb = cv2.cvtColor(frame_vis, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 7))
    plt.imshow(frame_vis_rgb)
    plt.title(f"Frame {current_frame} | {timestamp_str}")
    plt.axis("off")
    plt.show()

    current_frame += step

cap.release()

In [ ]:
current_frame = 0
row_id = 0
processed_frames = 0

In [ ]:
cap = cv2.VideoCapture(video_path)
frameno=0


while current_frame < total_frames:

    print("current_frame :",current_frame,"total_frames",total_frames)

    cap.set(cv2.CAP_PROP_POS_FRAMES, current_frame)
    ret, frame = cap.read()
    if not ret:
        break
    

    frame_no = current_frame
    timestamp_sec = frame_no / fps

    hrs = int(timestamp_sec // 3600)
    mins = int((timestamp_sec % 3600) // 60)
    secs = int(timestamp_sec % 60)

    timestamp_str = f"{hrs:02d}:{mins:02d}:{secs:02d}"

    print(timestamp_str)

    results = yolo_model.track(
    frame,
    persist=True,
    tracker="bytetrack.yaml",
    verbose=False
    )

    if not results:
        current_frame += step
        continue
    
    result = results[0]

    # print(" ===== === == == ==== ",result.names[class_id],round(confidence, 2)) 

    if result.boxes is None or len(result.boxes) == 0:
        
        current_frame += step
        continue
    
    # print(result.boxes)

    frame_emb_cache = None


    for box in result.boxes:
        if box.id is None:
            print("========  no detection!!! ================")
            continue

        track_id = int(box.id.item())

        class_id = int(box.cls.item())
        confidence = float(box.conf.item())
        print("frameno:",current_frame)
        print("track_id :", track_id,"===",class_id,"===",confidence)


        conf_threshold = 0.5

        if confidence < conf_threshold:
            print("========  Low Confidence!! ================")
            continue

        if track_id in seen_track_ids:
            print("========  already tracked!!! ================")
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())


        padding = 20

        x1 = max(0, x1 - padding)
        y1 = max(0, y1 - padding)
        x2 = min(width, x2 + padding)
        y2 = min(height, y2 + padding)



        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            print("========  small crop detection!!! ================")
            continue


        crop_rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    

        crop_inputs = siglip_processor(images=crop_rgb, return_tensors="pt")

        with torch.no_grad():
            crop_feats = siglip_model.get_image_features(**crop_inputs)

            if hasattr(crop_feats, "pooler_output"):
                crop_feats = crop_feats.pooler_output
            elif hasattr(crop_feats,"image_embeds"):
                crop_feats = crop_feats.image_embeds
            else:
                crop_feats = crop_feats
            
            crop_feats = crop_feats / crop_feats.norm(dim=-1, keepdim=True)

        crop_vec = crop_feats.squeeze(0).cpu().numpy()

        crop_emb_index = len(crop_embeddings)
        crop_embeddings.append(crop_vec)



        if frame_emb_cache is None:
            frame_inputs = siglip_processor(images=frame_rgb, return_tensors="pt")

            with torch.no_grad():
                frame_feats = siglip_model.get_image_features(**frame_inputs)

                if hasattr(frame_feats, "pooler_output"):
                    frame_feats = frame_feats.pooler_output
                elif hasattr(frame_feats,"image_embeds"):
                    frame_feats = frame_feats.image_embeds
                else:
                    frame_feats = frame_feats
            
                frame_feats = frame_feats / frame_feats.norm(dim=-1, keepdim=True)

            frame_emb_cache = frame_feats.squeeze(0).cpu().numpy()

            frame_emb_index = len(frame_embeddings)
            frame_embeddings.append(frame_emb_cache)
        else:
            frame_emb_index = len(frame_embeddings) - 1

        metadata_rows.append({
            "row_id": row_id,
            "track_id": track_id,
            "frame_no": frame_no,
            "timestamp": timestamp_str,
            "timestamp_sec": round(timestamp_sec, 2),
            "yolo_class": result.names[class_id],
            "confidence": round(confidence, 4),
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "crop_emb_index": crop_emb_index,
            "frame_emb_index": frame_emb_index
        })


        fig, ax = plt.subplots(figsize=(6, 4))
        ax.imshow(crop_rgb)
        ax.axis('off')
        display(fig)
        plt.close(fig)


        seen_track_ids.add(track_id)
        row_id += 1
    

    # frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # frame_emb_cache = None

    # fig, ax = plt.subplots(figsize=(6, 4))
    # ax.imshow(frame_rgb)
    # ax.axis('off')
        
    # display(fig)
    # plt.close(fig)



    processed_frames += 1
    current_frame += step
    frameno +=1

print(frameno)

crop_embeddings = np.array(crop_embeddings, dtype=np.float32)
frame_embeddings = np.array(frame_embeddings, dtype=np.float32)



metadata_df = pd.DataFrame(metadata_rows)


np.save("../data/video_embeddings/crop_embeddings.npy", crop_embeddings)
np.save("../data/video_embeddings/frame_embeddings.npy", frame_embeddings)
metadata_df.to_csv("../data/meta_data/vid_metadata.csv", index=False)



cap.release()

    